# 04 — Sales Streaming → Iceberg (with Customer Enrichment, via Kafka)

**Pre-requisite:** Start the sales producer in a separate terminal:
```bash
# host machine (Kafka on localhost:29092)
uv run python sales_producer.py

# or inside Docker
KAFKA_BOOTSTRAP_SERVERS=kafka:9092 uv run python sales_producer.py
```

**What this notebook does:**
1. Loads `data/source/customers.csv` as a **static** broadcast reference
2. Creates Iceberg table `iceberg_catalog.my_database.sales_enriched`
3. Reads a stream of JSON messages from Kafka topic `sales-events`
4. Enriches each sale event with customer info via a **stream-static join**
5. Writes the enriched stream into Iceberg (append mode)
6. Lets you monitor row count and revenue in real time

> The producer publishes 2 events every 10 s.  
> Spark triggers every 15 s so you will see rows accumulate in batches.

**Schema after enrichment:**

| Column | Source |
|--------|--------|
| sale_id, customer_id, product_id, product_name, category, quantity, unit_price, total_amount, store_id, payment_method, sale_timestamp | sales stream |
| customer_name, email, city, state, membership_tier | customers.csv join |

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
import os

spark = SparkSession.builder.appName("Spark-Developer").master("local[*]").getOrCreate()

# Kafka bootstrap servers — set automatically in Docker; override for local dev
KAFKA_BOOTSTRAP_SERVERS = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:29092")
print(f"Kafka: {KAFKA_BOOTSTRAP_SERVERS}")

spark

---
## Step 1 — Load static customer reference

Broadcasted to every executor so the stream-static join is efficient.

In [ ]:
customers_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("./data/source/customers.csv")
)

customers_df.printSchema()
customers_df.show(truncate=False)

---
## Step 2 — Create Iceberg target table

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS iceberg_catalog.my_database.sales_enriched (
        sale_id         STRING,
        customer_id     STRING,
        product_id      STRING,
        product_name    STRING,
        category        STRING,
        quantity        INT,
        unit_price      DOUBLE,
        total_amount    DOUBLE,
        store_id        STRING,
        payment_method  STRING,
        sale_timestamp  STRING,
        customer_name   STRING,
        email           STRING,
        city            STRING,
        state           STRING,
        membership_tier STRING
    )
    USING iceberg
    TBLPROPERTIES ('format-version' = '2')
""")

print("Table ready: iceberg_catalog.my_database.sales_enriched")
spark.sql("DESCRIBE iceberg_catalog.my_database.sales_enriched").show()

---
## Step 3 — Define streaming schema and Kafka reader

In [ ]:
CHECKPOINT_DIR = "./.tmp/checkpoint_sales_enriched"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

sale_schema = StructType([
    StructField("sale_id",        StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("product_name",   StringType(),  True),
    StructField("category",       StringType(),  True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("total_amount",   DoubleType(),  True),
    StructField("store_id",       StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("timestamp",      StringType(),  True),
])

# Read from Kafka — each message value is a JSON-encoded sale event
raw_stream = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
         .option("subscribe", "sales-events")
         .option("startingOffsets", "latest")
         .load()
)

# Decode binary value → parse JSON → flatten columns
stream_df = (
    raw_stream
    .select(F.from_json(F.col("value").cast("string"), sale_schema).alias("data"))
    .select("data.*")
)

print("Is streaming:", stream_df.isStreaming)
stream_df.printSchema()

---
## Step 4 — Start streaming write with customer enrichment

Stream-static join: each micro-batch is left-joined against the broadcasted  
`customers_df` so we get customer info without re-reading CSV every batch.

> This cell returns immediately. The stream runs in the background.

In [ ]:
enriched_df = (
    stream_df
    .join(F.broadcast(customers_df), on="customer_id", how="left")
    .select(
        "sale_id",
        "customer_id",
        "product_id",
        "product_name",
        "category",
        "quantity",
        "unit_price",
        "total_amount",
        "store_id",
        "payment_method",
        F.col("timestamp").alias("sale_timestamp"),
        F.col("name").alias("customer_name"),
        "email",
        "city",
        "state",
        "membership_tier",
    )
)

query = (
    enriched_df.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_DIR)
    .trigger(processingTime="15 seconds")
    .toTable("iceberg_catalog.my_database.sales_enriched")
)

print(f"Stream started  — id: {query.id}")
print(f"Status         : {query.status}")

---
## Step 5 — Monitor (re-run this cell every ~15–30 s)

In [ ]:
print("Active :", query.isActive)
print("Status :", query.status)

if query.lastProgress:
    prog = query.lastProgress
    print(f"Batch  : {prog['batchId']}")
    print(f"Input  : {prog['numInputRows']} rows in last batch")

print()

# Row count
spark.sql("""
    SELECT COUNT(*) AS total_sales
    FROM iceberg_catalog.my_database.sales_enriched
""").show()

# Revenue by category
spark.sql("""
    SELECT
        category,
        COUNT(*)                          AS sales_count,
        ROUND(SUM(total_amount), 2)       AS total_revenue,
        ROUND(AVG(total_amount), 2)       AS avg_order_value
    FROM iceberg_catalog.my_database.sales_enriched
    GROUP BY category
    ORDER BY total_revenue DESC
""").show()

# Revenue by membership tier
spark.sql("""
    SELECT
        membership_tier,
        COUNT(DISTINCT customer_id)       AS unique_customers,
        COUNT(*)                          AS sales_count,
        ROUND(SUM(total_amount), 2)       AS total_revenue
    FROM iceberg_catalog.my_database.sales_enriched
    GROUP BY membership_tier
    ORDER BY total_revenue DESC
""").show()

---
## Step 6 — Stop the stream

In [ ]:
query.stop()
print("Stream stopped.")

# Final summary
spark.sql("""
    SELECT
        COUNT(*)                    AS total_sales,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(DISTINCT product_id)  AS unique_products,
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM iceberg_catalog.my_database.sales_enriched
""").show()